In [1]:
import sys
sys.path.append('../')

import numpy as np
from matplotlib import pyplot as plt
from qutip.qip.operations import rz, cz_gate
from tqdm import tqdm
from matplotlib.colors import LogNorm
import pytz, cmath, itertools
import scqubits.settings as settings
settings.OVERLAP_THRESHOLD = 0.3
from joblib import Parallel, delayed
import scipy.sparse as ssp
from sympy import symbols
import utils_2Q_gate_zp as ut
import pandas as pd
import scipy as sp
from multiprocessing import Pool
import qutip as qt
import multiprocessing as mp
from multiprocessing import Pool
import scqubits as scq
from sympy import symbols
import scipy.sparse as ssp
from datetime import datetime


In [2]:
truc1, truc_tot, charge_pick = 300, 300, True
truc_tot_2 = 14
folder = f'../../data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}_eket/'
hspace_0 = pd.read_csv(folder+ 'hspace_0.txt').to_numpy().flatten()
hspace_1 = pd.read_csv(folder+ 'hspace_1.txt').to_numpy().flatten()
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()
eval_tot = 2*np.pi* pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()
n_theta0_dress = 2*np.pi* pd.read_csv(folder+ 'n_theta0_dress.txt').to_numpy()
n_theta1_dress = 2*np.pi* pd.read_csv(folder+ 'n_theta1_dress.txt').to_numpy()

cz300_se_3ncut= pd.read_csv('data/data_cz_3ncut_truc1=300_select.txt')
params = cz300_se_3ncut[['tg', 'drive_amp', 'detune']].to_numpy()[:1]

In [22]:
cz300_se_3ncut= pd.read_csv('data/data_cz_3ncut_truc1=300_select.txt')
params = cz300_se_3ncut[['tg', 'drive_amp', 'detune']].to_numpy()
params[[0, -1],:]

array([[2.00471530e+01, 1.94350000e-02, 7.27300000e-03],
       [2.00001076e+02, 6.93600000e-03, 1.20520000e-02]])

In [3]:
eket_tot = pd.read_csv(folder+ 'eket_tot.txt').map(complex).to_numpy()
eket_tot = eket_tot[:truc_tot_2]

In [4]:
dim_0 = len(hspace_0)
dim_1 = len(hspace_1)
eval_tot = eval_tot[:truc_tot_2]
hspace_full = hspace_full[:truc_tot_2]
hspace_dress = np.arange(truc_tot_2)
n_theta0_dress = qt.Qobj(n_theta0_dress[np.ix_(hspace_dress, hspace_dress)])
n_theta1_dress = qt.Qobj(n_theta1_dress[np.ix_(hspace_dress, hspace_dress)])

In [5]:
gamma_decay_logi =  0 / 1600e3
gamma_dephase_logi = 0
gamma_decay_other = 0 / 2e3
gamma_dephase_other = 0 / 400
jump_t1   = []
jump_tphi = []

if charge_pick:
    gamma_decay   = [0, gamma_decay_other,  gamma_decay_logi]  + [gamma_decay_other]  * (dim_1-3)
    gamma_dephase = [0, gamma_dephase_other, gamma_dephase_logi] + [gamma_dephase_other] * (dim_1-3)
    qubit_a = True
    args = [dim_0, dim_1, gamma_decay, gamma_dephase, eket_tot, qubit_a]
    jump_op_a = Parallel(n_jobs=100)(delayed(ut.get_jump_op_charge_pick)(state, *args) for state in range(1,dim_0))
    qubit_a = False
    args = [dim_0, dim_1, gamma_decay, gamma_dephase, eket_tot, qubit_a]
    jump_op_b = Parallel(n_jobs=100)(delayed(ut.get_jump_op_charge_pick)(state, *args) for state in range(1,dim_1))
    jump_t1_list = np.array(jump_op_a)[:,0].tolist() + np.array(jump_op_b)[:,0].tolist()
    jump_tphi_list = np.array(jump_op_a)[:,1].tolist() + np.array(jump_op_b)[:,1].tolist()
    jump_t1_list = [qt.Qobj(matrix) for matrix in jump_t1_list]
    jump_tphi_list = [qt.Qobj(matrix) for matrix in jump_tphi_list]
else:
    gamma_decay   = [0, gamma_decay_other,  gamma_decay_logi]  + [gamma_decay_other]  * (truc1-3)
    gamma_dephase = [0, gamma_dephase_other, gamma_dephase_logi] + [gamma_dephase_other] * (truc1-3)
    args = [truc1, gamma_decay, gamma_dephase, eket_tot]
    jump_op = Parallel(n_jobs=100)(delayed(ut.get_jump_op)(state, *args) for state in range(1,truc1))
    jump_t1 = np.array(jump_op)[:,:2]
    jump_tphi = np.array(jump_op)[:,2:]
    jump_t1_list = [qt.Qobj(matrix) for row in jump_t1 for matrix in row]
    jump_tphi_list = [qt.Qobj(matrix) for row in jump_tphi for matrix in row]

In [17]:
logi_state = ['0-0', '0-2', '2-0', '2-2']
W_20_50 = np.abs(eval_tot[hspace_full.index('2-0')] - eval_tot[hspace_full.index('5-0')])
H0 = qt.Qobj(np.diag(eval_tot))
logi_idx = [hspace_full.index(state) for state in logi_state]
H_qbt_drive = [H0, [n_theta1_dress, ut.drive_gauss_A] ]

num_cpus, n_job = 100, 1
max_steps = 1e-4

tg, drive_amp, detune = params[0] # Independent arguments that can be optimized over
pulse_args = {'drive_amp_A': drive_amp,
            'drive_freq_A': W_20_50 + 2*np.pi*detune,
            'gate_time': tg}
tlist = np.linspace(0, tg, num=3*int(tg))  # total time
options =qt.Options(max_step=max_steps, nsteps=1e4, num_cpus=1 )
print(hspace_full)
print(logi_idx)

['0-0', '0-1', '1-0', '0-2', '2-0', '0-4', '4-0', '1-1', '0-5', '2-1', '5-0', '1-2', '0-8', '2-2']
[0, 3, 4, 13]


In [18]:
c_op_list = []
p = qt.propagator( H=H_qbt_drive,
                        t=tlist,
                        c_op_list=c_op_list,
                        options=options,
                        args=pulse_args,
                        num_cpus=num_cpus,
                        parallel=True,
                        )[-1]  # get the propagator at the final time step
p0_kraus = qt.to_kraus(qt.to_super(p))
p0_kraus = [ut.truncate_2(i, logi_idx) for i in p0_kraus]
p0_kraus_zz = ut.cz_phase_correct(p0_kraus)
p0_super_2 = qt.kraus_to_super(p0_kraus_zz)
f_noise = qt.metrics.average_gate_fidelity(p0_super_2, target=cz_gate())
print(f'c_op_list.shape = {np.shape(c_op_list)}', ', num_cpus =', num_cpus)
print('max_steps =', max_steps)
print('fidelity (qutip) =', np.round(np.log10(1-f_noise), 10))

args = [H_qbt_drive, W_20_50, max_steps, num_cpus, c_op_list, logi_idx ]
f_ideal = Parallel(n_jobs=n_job, verbose=0)(delayed(ut.cz_fidelity_log_noise)(args_indep, *args)
                                            for args_indep in params)
print('fidelity (ZL) =', np.round(f_ideal[0], 10))

c_op_list.shape = (0,) , num_cpus = 100
max_steps = 0.0001
fidelity (qutip) = -0.5419679262
fidelity (ZL) = -0.5419679262


In [19]:
num_cpus = 100
c_op_list = jump_t1_list + jump_tphi_list
p = qt.propagator( H=H_qbt_drive,
                        t=tlist,
                        c_op_list=c_op_list,
                        options=options,
                        args=pulse_args,
                        num_cpus=num_cpus,
                        parallel=True,
                        )[-1]  # get the propagator at the final time step
p0_kraus = qt.to_kraus(qt.to_super(p))
p0_kraus = [ut.truncate_2(i, logi_idx) for i in p0_kraus]
p0_kraus_zz = ut.cz_phase_correct(p0_kraus)
p0_super_2 = qt.kraus_to_super(p0_kraus_zz)
f_noise = qt.metrics.average_gate_fidelity(p0_super_2, target=cz_gate())
print(f'c_op_list.shape = {np.shape(c_op_list)}', ', num_cpus =', num_cpus)
print('max_steps =', max_steps)
print('fidelity (qutip) =', np.round(np.log10(1-f_noise), 10))

args = [H_qbt_drive, W_20_50, max_steps, num_cpus, c_op_list, logi_idx ]
f_ideal = Parallel(n_jobs=n_job, verbose=0)(delayed(ut.cz_fidelity_log_noise)(args_indep, *args)
                                            for args_indep in params)
print('fidelity (ZL) =', np.round(f_ideal[0], 10))

c_op_list.shape = (614, 14, 14) , num_cpus = 100
max_steps = 0.0001
fidelity (qutip) = -0.54195942
fidelity (ZL) = -0.5419603152


In [ ]:
num_cpus = 1
c_op_list = jump_t1_list + jump_tphi_list
p = qt.propagator( H=H_qbt_drive,
                        t=tlist,
                        c_op_list=c_op_list,
                        options=options,
                        args=pulse_args,
                        num_cpus=num_cpus,
                        parallel=True,
                        )[-1]  # get the propagator at the final time step
p0_kraus = qt.to_kraus(qt.to_super(p))
p0_kraus = [ut.truncate_2(i, logi_idx) for i in p0_kraus]
p0_kraus_zz = ut.cz_phase_correct(p0_kraus)
p0_super_2 = qt.kraus_to_super(p0_kraus_zz)
f_noise = qt.metrics.average_gate_fidelity(p0_super_2, target=cz_gate())
print(f'c_op_list.shape = {np.shape(c_op_list)}', ', num_cpus =', num_cpus)
print('max_steps =', max_steps)
print('fidelity (qutip) =', np.round(np.log10(1-f_noise), 10))

args = [H_qbt_drive, W_20_50, max_steps, num_cpus, c_op_list, logi_idx ]
f_ideal = Parallel(n_jobs=n_job, verbose=0)(delayed(ut.cz_fidelity_log_noise)(args_indep, *args)
                                            for args_indep in params)
print('fidelity (ZL) =', np.round(f_ideal[0], 10))